In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

In [3]:
with engine.connect() as connection:
    print("Connected!")

Connected!


In [4]:
dataset_query = """
select
    *
from analytics.fraud_ml_features
limit 5
"""

dataset = pd.read_sql(dataset_query, engine)

dataset

,transaction_key,transaction_timestamp,amount,mcc_key,use_chip,merchant_id,transaction_hour,day_of_week,is_weekend,user_previous_transaction_count,user_previous_avg_amount,amount_vs_user_avg,card_previous_transaction_count,card_previous_avg_amount,amount_vs_card_avg,is_fraud
0,9564726,2010-01-01 13:25:00,10.24,4,Swipe Transaction,21491,13.0,5.0,0,0,NaN,NaN,0,NaN,NaN,False
1,8733454,2010-01-02 06:26:00,138.32,16,Swipe Transaction,60569,6.0,6.0,1,2,53.88,84.44,2,53.88,84.44,False
2,3536380,2010-01-02 07:28:00,181.83,53,Swipe Transaction,98050,7.0,6.0,1,4,-40.98,222.81,4,-40.98,222.81,False
3,12683385,2010-01-02 13:10:00,154.84,81,Swipe Transaction,71883,13.0,6.0,1,5,3.58,151.26,5,3.58,151.26,False
4,5201105,2010-01-02 13:24:00,-191.00,81,Swipe Transaction,71883,13.0,6.0,1,6,28.79,-219.79,6,28.79,-219.79,False


In [5]:
dataset_info_query = """
select
    min(transaction_timestamp) as min_date,
    max(transaction_timestamp) as max_date,
    count(*) as number_of_transactions
from analytics.fraud_ml_features
"""

dataset_info = pd.read_sql(dataset_info_query, engine)

dataset_info

,min_date,max_date,number_of_transactions
0,2010-01-01 00:01:00,2019-10-31 23:57:00,8914963


In [6]:
fraud_by_year_query = """
select
    extract(year from transaction_timestamp)::int as year,
    count(*) as transactions,
    count(*) filter (where is_fraud = true) as fraud_transactions,
    round(count(*) filter (where is_fraud = true)::numeric / count(*) * 100, 4) as fraud_rate
from analytics.fraud_ml_features
group by extract(year from transaction_timestamp)
order by year
"""

fraud_by_year = pd.read_sql(fraud_by_year_query, engine)

fraud_by_year

,year,transactions,fraud_transactions,fraud_rate
0,2010,831529,2573,0.3094
1,2011,863428,37,0.0043
2,2012,885421,923,0.1042
3,2013,907304,1337,0.1474
4,2014,915073,664,0.0726
5,2015,930224,2189,0.2353
6,2016,932762,2448,0.2624
7,2017,937284,172,0.0184
8,2018,934599,1629,0.1743
9,2019,777339,1360,0.1750


In [7]:
fraud_by_month_query = """
select
    date_trunc('month', transaction_timestamp) as month,
    count(*) as transactions,
    count(*) filter (where is_fraud = true) as fraud_transactions,
    round(count(*) filter (where is_fraud = true)::numeric / count(*) * 100, 4) as fraud_rate
from analytics.fraud_ml_features
group by date_trunc('month', transaction_timestamp)
order by month
"""

fraud_by_month = pd.read_sql(fraud_by_month_query, engine)

fraud_by_month

,month,transactions,fraud_transactions,fraud_rate
0,2010-01-01,68044,107,0.1573
1,2010-02-01,62816,259,0.4123
2,2010-03-01,69202,261,0.3772
3,2010-04-01,66729,237,0.3552
4,2010-05-01,70210,274,0.3903
...,...,...,...,...
113,2019-06-01,77089,132,0.1712
114,2019-07-01,79891,104,0.1302
115,2019-08-01,79513,165,0.2075
116,2019-09-01,76977,90,0.1169


In [8]:
fraud_by_month["fraud_transactions"].describe()

count    118.000000
mean     112.983051
std       99.149717
min        0.000000
25%        0.000000
50%      116.000000
75%      192.750000
max      423.000000
Name: fraud_transactions, dtype: float64

In [9]:
print("Months with 0 fraud:",
      (fraud_by_month["fraud_transactions"] == 0).sum())

print("Months with < 10 fraud:",
      (fraud_by_month["fraud_transactions"] < 10).sum())

print("Min fraud rate:",
      fraud_by_month["fraud_rate"].min())

print("Max fraud rate:",
      fraud_by_month["fraud_rate"].max())

Months with 0 fraud: 39
Months with < 10 fraud: 40
Min fraud rate: 0.0
Max fraud rate: 0.5278


In [10]:
zero_fraud_months = fraud_by_month[
    fraud_by_month["fraud_transactions"] == 0
]

zero_fraud_months

,month,transactions,fraud_transactions,fraud_rate
13,2011-02-01,65223,0,0.0
14,2011-03-01,72224,0,0.0
15,2011-04-01,70777,0,0.0
16,2011-05-01,72667,0,0.0
17,2011-06-01,71162,0,0.0
18,2011-07-01,73440,0,0.0
19,2011-08-01,74420,0,0.0
20,2011-09-01,71418,0,0.0
21,2011-10-01,73877,0,0.0
22,2011-11-01,71564,0,0.0


In [11]:
train_query = """
(
    select *
    from analytics.fraud_ml_features
    where transaction_timestamp < '2018-01-01'
      and is_fraud = true
)
union all
(
    select *
    from analytics.fraud_ml_features
    where transaction_timestamp < '2018-01-01' and is_fraud = false
    order by random()
    limit 300000
)
"""

train_df = pd.read_sql(train_query, engine)

train_df.shape

(310343, 16)

In [12]:
train_df["is_fraud"].value_counts()

is_fraud
False    300000
True      10343
Name: count, dtype: int64

In [13]:
train_df["is_fraud"].value_counts(normalize=True)

is_fraud
False    0.966672
True     0.033328
Name: proportion, dtype: float64

In [14]:
train_df.columns.tolist()

['transaction_key',
 'transaction_timestamp',
 'amount',
 'mcc_key',
 'use_chip',
 'merchant_id',
 'transaction_hour',
 'day_of_week',
 'is_weekend',
 'user_previous_transaction_count',
 'user_previous_avg_amount',
 'amount_vs_user_avg',
 'card_previous_transaction_count',
 'card_previous_avg_amount',
 'amount_vs_card_avg',
 'is_fraud']

In [15]:
X_train = train_df.drop(
    columns=[
        "transaction_key",
        "transaction_timestamp",
        "is_fraud"
    ]
)

y_train = train_df["is_fraud"]

In [16]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (310343, 13)
y_train: (310343,)


In [17]:
val_query = """
select *
from analytics.fraud_ml_features
where transaction_timestamp >= '2018-01-01' and transaction_timestamp < '2019-01-01'
"""

val_df = pd.read_sql(val_query, engine)

val_df.shape

(934599, 16)

In [18]:
X_val = val_df.drop(
    columns=[
        "transaction_key",
        "transaction_timestamp",
        "is_fraud"
    ]
)

y_val = val_df["is_fraud"]

In [19]:
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nValidation target:")
print(y_val.value_counts())

print("\nValidation proportions:")
print(y_val.value_counts(normalize=True))

X_val: (934599, 13)
y_val: (934599,)

Validation target:
is_fraud
False    932970
True       1629
Name: count, dtype: int64

Validation proportions:
is_fraud
False    0.998257
True     0.001743
Name: proportion, dtype: float64


## Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression
lrg = LogisticRegression()
lrg.fit(X_train, y_train)
print("Training set score: {:.2f}".format(lrg.score(X_train, y_train)))
print("Validation set score: {:.2f}".format(lrg.score(X_val, y_val)))

ValueError: could not convert string to float: 'Online Transaction'

In [21]:
X_train.dtypes

amount                             float64
mcc_key                              int64
use_chip                               str
merchant_id                          int64
transaction_hour                   float64
day_of_week                        float64
is_weekend                           int64
user_previous_transaction_count      int64
user_previous_avg_amount           float64
amount_vs_user_avg                 float64
card_previous_transaction_count      int64
card_previous_avg_amount           float64
amount_vs_card_avg                 float64
dtype: object

In [22]:
X_train["use_chip"].value_counts(dropna=False)

use_chip
Swipe Transaction     184029
Chip Transaction       83201
Online Transaction     43113
Name: count, dtype: int64

In [23]:
from sklearn.preprocessing import OneHotEncoder

categorical_columns = X_train.select_dtypes(include=['str']).columns
encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(X_train[categorical_columns])

encoded_X_train = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_columns)
)

final_X_train = pd.concat(
    [X_train.drop(columns=categorical_columns), encoded_X_train],
    axis=1
)
final_X_train.head()

,amount,mcc_key,merchant_id,transaction_hour,day_of_week,is_weekend,user_previous_transaction_count,user_previous_avg_amount,amount_vs_user_avg,card_previous_transaction_count,card_previous_avg_amount,amount_vs_card_avg,use_chip_Chip Transaction,use_chip_Online Transaction,use_chip_Swipe Transaction
0,248.36,66,76639,9.0,1.0,0,6750,81.11,167.25,1636,65.76,182.60,0.0,1.0,0.0
1,44.57,16,60569,13.0,1.0,0,6754,81.19,-36.62,1639,66.17,-21.60,0.0,1.0,0.0
2,64.93,3,75393,14.0,2.0,0,6758,81.22,-16.29,1640,66.16,-1.23,0.0,1.0,0.0
3,124.59,10,99370,15.0,2.0,0,6759,81.22,43.37,1641,66.15,58.44,0.0,1.0,0.0
4,97.34,33,80770,15.0,2.0,0,6760,81.22,16.12,1642,66.19,31.15,0.0,1.0,0.0


In [24]:
categorical_columns = X_val.select_dtypes(include=['str']).columns

encoded_data_val = encoder.transform(X_val[categorical_columns])

encoded_X_val = pd.DataFrame(
    encoded_data_val,
    columns=encoder.get_feature_names_out(categorical_columns)
)

final_X_val = pd.concat(
    [X_val.drop(columns=categorical_columns), encoded_X_val],
    axis=1
)

In [25]:
final_X_val.head()

,amount,mcc_key,merchant_id,transaction_hour,day_of_week,is_weekend,user_previous_transaction_count,user_previous_avg_amount,amount_vs_user_avg,card_previous_transaction_count,card_previous_avg_amount,amount_vs_card_avg,use_chip_Chip Transaction,use_chip_Online Transaction,use_chip_Swipe Transaction
0,123.97,2,61195,5.0,1.0,0,9208,80.80,43.17,4405,98.11,25.86,0.0,0.0,1.0
1,29.84,6,93412,20.0,2.0,0,9210,80.81,-50.97,748,63.81,-33.97,1.0,0.0,0.0
2,39.02,7,20519,6.0,3.0,0,9212,80.81,-41.79,749,63.76,-24.74,1.0,0.0,0.0
3,129.34,4,46284,6.0,5.0,0,9214,80.80,48.54,2075,64.96,64.38,1.0,0.0,0.0
4,148.50,6,93412,9.0,5.0,0,9216,80.80,67.70,4407,98.12,50.38,0.0,0.0,1.0


In [26]:
from sklearn.linear_model import LogisticRegression
lrg = LogisticRegression()
lrg.fit(final_X_train, y_train)
print("Training set score: {:.2f}".format(lrg.score(final_X_train, y_train)))
print("Validation set score: {:.2f}".format(lrg.score(final_X_val, y_val)))

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [27]:
final_X_train.isna().sum()

amount                               0
mcc_key                              0
merchant_id                          0
transaction_hour                     0
day_of_week                          0
is_weekend                           0
user_previous_transaction_count      0
user_previous_avg_amount            38
amount_vs_user_avg                  38
card_previous_transaction_count      0
card_previous_avg_amount           125
amount_vs_card_avg                 125
use_chip_Chip Transaction            0
use_chip_Online Transaction          0
use_chip_Swipe Transaction           0
dtype: int64

In [28]:
final_X_train = final_X_train.fillna(final_X_train.median())

In [29]:
final_X_val = final_X_val.fillna(final_X_train.median())

In [30]:
final_X_train.isna().sum()

amount                             0
mcc_key                            0
merchant_id                        0
transaction_hour                   0
day_of_week                        0
is_weekend                         0
user_previous_transaction_count    0
user_previous_avg_amount           0
amount_vs_user_avg                 0
card_previous_transaction_count    0
card_previous_avg_amount           0
amount_vs_card_avg                 0
use_chip_Chip Transaction          0
use_chip_Online Transaction        0
use_chip_Swipe Transaction         0
dtype: int64

In [31]:
final_X_val.isna().sum()

amount                             0
mcc_key                            0
merchant_id                        0
transaction_hour                   0
day_of_week                        0
is_weekend                         0
user_previous_transaction_count    0
user_previous_avg_amount           0
amount_vs_user_avg                 0
card_previous_transaction_count    0
card_previous_avg_amount           0
amount_vs_card_avg                 0
use_chip_Chip Transaction          0
use_chip_Online Transaction        0
use_chip_Swipe Transaction         0
dtype: int64

In [32]:
lrg = LogisticRegression()
lrg.fit(final_X_train, y_train)
print("Training set score: {:.2f}".format(lrg.score(final_X_train, y_train)))
# Accuracy
print("Validation set score: {:.2f}".format(lrg.score(final_X_val, y_val)))

Training set score: 0.97
Validation set score: 1.00


/Users/martyna/Documents/Projects/finsight-fintech-analytics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [33]:
pred_y = lrg.predict(final_X_val)
pred_y.sum()

np.int64(716)

In [34]:
y_val.sum()

np.int64(1629)

In [35]:
correctly_detected_frauds = ((y_val == True) & (pred_y == True)).sum()

print("Correctly detected frauds:", correctly_detected_frauds)

Correctly detected frauds: 4


#### Recall

In [36]:
from sklearn.metrics import recall_score

recall = recall_score(y_val, pred_y)
print("Recall: {:.4f}".format(recall))

Recall: 0.0025


#### Precision

In [37]:
from sklearn.metrics import precision_score

precision = precision_score(y_val, pred_y)
print("Precision: {:.4f}".format(precision))

Precision: 0.0056
